# Four Qubit Chip Design

In [1]:
%reload_ext autoreload
%autoreload 2

## Prerequisite

In [2]:
import os
import numpy as np
import pandas as pd
from collections import OrderedDict

from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, Headings

from qiskit_metal.qlibrary.qubits.transmon_pocket_cl import TransmonPocketCL
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.anchored_path import RouteAnchors
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.terminations.launchpad_wb_coupled import LaunchpadWirebondCoupled

## Four Qubit Design

We will aim for the following parameters on our chip;

- qubit frequency => 4.8, 5, 5.2, 5.4 GHz
- qubit anharmonicity => 300 MHz
- qubit-bus (g) => 80 MHz
- qubit-readout (chi) => 1 MHz
- bus frequency 
    - Q1 (bus_12, bus_14) => 5.8, 6.0 GHz
    - Q2 (bus_23, bus_12) => 6.2, 5.8 GHz
    - Q3 (bus_34, bus_23) => 6.4, 6.2 GHz
    - Q4 (bus_14, bus_34) => 6.0, 6.4 GHz
- readout frequencies => 6.8, 7, 7.2, 7.4 GHz
- readout Q_external => 2000

In [3]:
design = designs.DesignPlanar()

gui = MetalGUI(design)

In [4]:
design.overwrite_enabled = True

In [5]:
design.variables['cpw_width'] = '10 um'
design.variables['cpw_gap'] = '6 um'
design._chips['main']['size']['size_x'] = '9mm'
design._chips['main']['size']['size_y'] = '6.5mm'

We need 4 transmons with 3 connection pads each and a chargeline. Let's explore the options of one transmon

In [6]:
TransmonPocketCL.get_template_options(design)

{'pos_x': '0.0um',
 'pos_y': '0.0um',
 'orientation': '0.0',
 'chip': 'main',
 'layer': '1',
 'connection_pads': {},
 '_default_connection_pads': {'pad_gap': '15um',
  'pad_width': '125um',
  'pad_height': '30um',
  'pad_cpw_shift': '5um',
  'pad_cpw_extent': '25um',
  'cpw_width': 'cpw_width',
  'cpw_gap': 'cpw_gap',
  'cpw_extend': '100um',
  'pocket_extent': '5um',
  'pocket_rise': '65um',
  'loc_W': '+1',
  'loc_H': '+1'},
 'pad_gap': '30um',
 'inductor_width': '20um',
 'pad_width': '455um',
 'pad_height': '90um',
 'pocket_width': '650um',
 'pocket_height': '650um',
 'make_CL': True,
 'cl_gap': '6um',
 'cl_width': '10um',
 'cl_length': '20um',
 'cl_ground_gap': '6um',
 'cl_pocket_edge': '0',
 'cl_off_center': '50um',
 'hfss_wire_bonds': False,
 'q3d_wire_bonds': False,
 'aedt_q3d_wire_bonds': False,
 'aedt_hfss_wire_bonds': False,
 'hfss_inductance': '10nH',
 'hfss_capacitance': 0,
 'hfss_resistance': 0,
 'hfss_mesh_kw_jj': 7e-06,
 'q3d_inductance': '10nH',
 'q3d_capacitance': 0,
 

We want to change the `pad_width` for these transmons, as well as define the 3 connection pads and chargeline.

To apply the same modifications to all 4 transmons, we define a single option-dictionary to pass to all transmons at the monent of creation

In [7]:
offset_tm = 69  #we the transmon slightly out of center-line

q1 = TransmonPocketCL(design, 'Q1', options = dict(
    pos_x='-2420um', 
    pos_y=f'{offset_tm}um', 
    orientation = '180',
    gds_cell_name='FakeJunction_01',
    cl_off_center = '-50um',
    cl_pocket_edge = '180',
    connection_pads=dict(
        readout = dict(loc_W=+1, loc_H=-1, pad_width='70um', cpw_extend =  '50um'), 
        bus_12 = dict(loc_W=-1, loc_H=-1, pad_width='125um', cpw_extend = '50um'),
        bus_14 = dict(loc_W=-1, loc_H=+1, pad_width='110um', cpw_extend = '50um')
    )))
q2 = TransmonPocketCL(design, 'Q2', options = dict(
    pos_x='0um', 
    pos_y='+857.6um', 
    orientation = '90',
    gds_cell_name='FakeJunction_01',
    cl_off_center = '-50um',
    cl_pocket_edge = '180',
    connection_pads=dict(
        readout = dict(loc_W=+1, loc_H=-1, pad_width='70um', cpw_extend = '50um'), 
        bus_23 = dict(loc_W=-1, loc_H=-1, pad_width='125um', cpw_extend = '50um'),
        bus_12 = dict(loc_W=-1, loc_H=+1, pad_width='110um', cpw_extend = '50um')
    )))

q3 = TransmonPocketCL(design, 'Q3', options = dict(
    pos_x='+2420um', 
    pos_y=f'{offset_tm}um',
    gds_cell_name='FakeJunction_01',
    cl_off_center = '-50um',
    cl_pocket_edge = '180', 
    connection_pads=dict(
        readout = dict(loc_W=+1, loc_H=-1, pad_width='70um', cpw_extend = '50um'), 
        bus_34 = dict(loc_W=-1, loc_H=-1, pad_width='125um', cpw_extend = '50um'),
        bus_23 = dict(loc_W=-1, loc_H=+1, pad_width='110um', cpw_extend = '50um')
    )))

q4 = TransmonPocketCL(design, 'Q4', options = dict(
    pos_x='0um', 
    pos_y='-857.6um',
    orientation = '270', 
    gds_cell_name='FakeJunction_01',
    cl_off_center = '-50um',
    cl_pocket_edge = '180',
    connection_pads=dict(
        readout = dict(loc_W=+1, loc_H=-1, pad_width='70um', cpw_extend = '50um'), 
        bus_14 = dict(loc_W=-1, loc_H=-1, pad_width='125um', cpw_extend = '50um'),
        bus_34 = dict(loc_W=-1, loc_H=+1, pad_width='110um', cpw_extend = '50um')
    )))

gui.rebuild()
gui.autoscale()

We can now create the 4 transmons by specifying the desired coordinates and orientations.

Let's now connect the transmons with tranismission lines. We want to have an "exact length" transmission line, so we will use the `RouteMeander`. Let's first observe what are the default options

In [8]:
RouteMeander.get_template_options(design)

{'chip': 'main',
 'layer': '1',
 'pin_inputs': {'start_pin': {'component': '', 'pin': ''},
  'end_pin': {'component': '', 'pin': ''}},
 'fillet': '0',
 'lead': {'start_straight': '0mm',
  'end_straight': '0mm',
  'start_jogged_extension': '',
  'end_jogged_extension': ''},
 'total_length': '7mm',
 'trace_width': 'cpw_width',
 'meander': {'spacing': '200um', 'asymmetry': '0um'},
 'snap': 'true',
 'prevent_short_edges': 'true',
 'hfss_wire_bonds': False,
 'q3d_wire_bonds': False,
 'aedt_q3d_wire_bonds': False,
 'aedt_hfss_wire_bonds': False}

We want to globally override the default lead (straight initial segment leaving the transmon) and the default fillet (corner rounding radius). Let's collect this information in one dictionary

In [9]:
from qiskit_metal.analyses.em.cpw_calculations import guided_wavelength

def find_resonator_length(frequency, line_width, line_gap, N): 
    #frequency in GHz
    #line_width/line_gap in um
    #N -> 2 for lambda/2, 4 for lambda/4
    #substrate dimensions and properties already set
    
    [lambdaG, etfSqrt, q] = guided_wavelength(frequency*10**9, line_width*10**-6,
                                              line_gap*10**-6, 750*10**-6, 200*10**-9, 11.9)
    return str(lambdaG/N*10**3)+" mm"

In [10]:
q1.pins.keys()

dict_keys(['readout', 'bus_12', 'bus_14', 'Charge_Line'])

In [11]:
bus_12 = find_resonator_length(frequency=5.8, line_width=10, line_gap=6, N=2)
bus_14 = find_resonator_length(frequency=6.0, line_width=10, line_gap=6, N=2)
bus_23 = find_resonator_length(frequency=6.2, line_width=10, line_gap=6, N=2)
bus_34 = find_resonator_length(frequency=6.4, line_width=10, line_gap=6, N=2)

bus_12 = str(round(float(bus_12.split()[0]), 2)) + 'mm'
bus_14 = str(round(float(bus_14.split()[0]), 2)) + 'mm'
bus_23 = str(round(float(bus_23.split()[0]), 2)) + 'mm'
bus_34 = str(round(float(bus_34.split()[0]), 2)) + 'mm'

print('bus_12 length: ' + bus_12)
print('bus_14 length: ' + bus_14)
print('bus_23 length: ' + bus_23)
print('bus_34 length: ' + bus_34)

bus_12 length: 10.31mm
bus_14 length: 9.97mm
bus_23 length: 9.64mm
bus_34 length: 9.34mm


In [12]:
fillet='99.99um'
cpw_options = Dict(
    lead=Dict(
        start_straight='100um',
        end_straight='250um'),
    fillet=fillet
    )

We then want each transmission line to be connected to different pins and to have different lengths and asymmetry w.r.t their centerline. Let's collect this information in other dictionaries. Before doing that, to manage the dictionaries in a simpler way, we redefine the `RouteMeander` signature by wrapping it into a convenience method named `connect`

In [13]:
def connect(cpw_name: str, pin1_comp_name: str, pin1_comp_pin: str, pin2_comp_name: str, pin2_comp_pin: str,
            length: str, asymmetry='0 um'):
    """Connect two pins with a CPW."""
    myoptions = Dict(
        pin_inputs=Dict(
            start_pin=Dict(
                component=pin1_comp_name,
                pin=pin1_comp_pin),
            end_pin=Dict(
                component=pin2_comp_name,
                pin=pin2_comp_pin)),
        total_length=length)
    myoptions.update(cpw_options)
    myoptions.meander.asymmetry = asymmetry
    return RouteMeander(design, cpw_name, myoptions)

We can now proceed and define the meanders following the signature: `connect(cpw_name, pin1_comp_name, pin1_comp_pin, pin2_comp_name, pin2_comp_pin, length, asymmetry)`

In [14]:
asym = 500
cpw1 = connect('cpw1', 'Q1', 'bus_12', 'Q2', 'bus_12', bus_12, f'+{asym+0.75*offset_tm}um')
cpw2 = connect('cpw2', 'Q3', 'bus_23', 'Q2', 'bus_23', bus_23, f'-{asym+0.75*offset_tm}um')
cpw3 = connect('cpw3', 'Q3', 'bus_34', 'Q4', 'bus_34', bus_34, f'+{asym-1.25*offset_tm}um')
cpw4 = connect('cpw4', 'Q1', 'bus_14', 'Q4', 'bus_14', bus_14, f'-{asym-1.25*offset_tm}um')

gui.rebuild()
gui.autoscale()

### Let's now connect the core elements to the launchpads

First we setup the launchpad location and orientation

In [15]:
# V1 - Corners
launch_cl_q1 = LaunchpadWirebond(design, 'Launch_CL_Q1', options = dict(pos_x='-3545um', pos_y='-2812um', orientation='90', lead_length='0um'))
launch_cl_q2 = LaunchpadWirebond(design, 'Launch_CL_Q2', options = dict(pos_x='-3545um', pos_y='2812um', orientation='270', lead_length='0um'))
launch_cl_q3 = LaunchpadWirebond(design, 'Launch_CL_Q3', options = dict(pos_x='3545um', pos_y='2812um', orientation='270', lead_length='0um'))
launch_cl_q4 = LaunchpadWirebond(design, 'Launch_CL_Q4', options = dict(pos_x='3545um', pos_y='-2812um', orientation='90', lead_length='0um'))

# V2
launch_ro_q1 = LaunchpadWirebondCoupled(design, 'Launch_RO_Q1', options = dict(pos_x='-4020um', pos_y='0', orientation='0', lead_length='30um'))
launch_ro_q2 = LaunchpadWirebondCoupled(design, 'Launch_RO_Q2', options = dict(pos_x='990um', pos_y='2812um', orientation='270', lead_length='30um'))
launch_ro_q3 = LaunchpadWirebondCoupled(design, 'Launch_RO_Q3', options = dict(pos_x='4020um', pos_y='0', orientation='180', lead_length='30um'))
launch_ro_q4 = LaunchpadWirebondCoupled(design, 'Launch_RO_Q4', options = dict(pos_x='-990um', pos_y='-2812um', orientation='90', lead_length='30um'))

gui.rebuild()
gui.autoscale()

Then we route. First the V2 launchpads - Exchange Coupler Lines to Edges

Connected to the readout resonators of each qubit

Used for dispersive readout — you send a probe tone and measure the reflected/transmitted signal to determine the qubit state without directly measuring it

In [ ]:
readout1 = find_resonator_length(frequency=6.8, line_width=10, line_gap=6, N=2)
readout2 = find_resonator_length(frequency=7.0, line_width=10, line_gap=6, N=2)
readout3 = find_resonator_length(frequency=7.2, line_width=10, line_gap=6, N=2)
readout4 = find_resonator_length(frequency=7.4, line_width=10, line_gap=6, N=2)

readout1 = str(round(float(readout1.split()[0]), 2)) + 'mm'
readout2 = str(round(float(readout2.split()[0]), 2)) + 'mm'
readout3 = str(round(float(readout3.split()[0]), 2)) + 'mm'
readout4 = str(round(float(readout4.split()[0]), 2)) + 'mm'

print('readout1 length: ' + readout1)
print('readout2 length: ' + readout2)
print('readout3 length: ' + readout3)
print('readout4 length: ' + readout4)

readout1 length: 8.793778093280148 mm
readout2 length: 8.542527289347825 mm
readout3 length: 8.305234863383255 mm
readout4 length: 8.080769055010833 mm


In [17]:
asym = 150
cpw_options = Dict(
    lead=Dict(
        start_straight='430um',
        end_straight='0um'),
    fillet=fillet
    )
readout1 = connect('Readout1', 'Q1', 'readout', 'Launch_RO_Q1', 'tie', '8.7 mm', f'+{asym}um')
readout3 = connect('Readout3', 'Q3', 'readout', 'Launch_RO_Q3', 'tie', '8.3 mm', f'+{asym}um')

asym = 200
cpw_options = Dict(
    lead=Dict(
        start_straight='535um',
        end_straight='0um'),
    fillet=fillet
    )
readout2 = connect('Readout2', 'Q2', 'readout', 'Launch_RO_Q2', 'tie', '8.5 mm', f'+{asym}um')
readout4 = connect('Readout4', 'Q4', 'readout', 'Launch_RO_Q4', 'tie', '8.0 mm', f'+{asym}um')

gui.rebuild()
gui.autoscale()

Finally we route the V1 launchpads - Charge Lines to Corners

We create the transmission lines between the corner launchpads and the open to grounds

Connected to the charge lines of each qubit

Used for single-qubit control — sending microwave pulses to drive qubit transitions

In [18]:
from collections import OrderedDict
jogsA_in = OrderedDict()
jogsA_in[0] = ["L", '200um']

options_charge_line1 = {'pin_inputs': 
            {'start_pin': {'component': 'Q1', 'pin': 'Charge_Line'}, 
             'end_pin': {'component': 'Launch_CL_Q1', 'pin': 'tie'}},
            'lead': {'start_straight': '120um', 'end_straight': '225um','start_jogged_extension': jogsA_in},
            'fillet': fillet
            }
charge_line1 = RouteAnchors(design, 'Charge_Line1', options_charge_line1)

options_charge_line3 = {'pin_inputs': 
            {'start_pin': {'component': 'Q3', 'pin': 'Charge_Line'}, 
             'end_pin': {'component': 'Launch_CL_Q3', 'pin': 'tie'}},
            'lead': {'start_straight': '120um', 'end_straight': '225um', 'start_jogged_extension': jogsA_in},
            'fillet': fillet
            }
charge_line3 = RouteAnchors(design, 'Charge_Line3', options_charge_line3)

gui.rebuild()
gui.autoscale()

In [19]:
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder

jogsB_in = OrderedDict()
jogsB_in[0] = ["L", '300um']

anchors2c = OrderedDict()
anchors2c[0] = np.array([-2, 2.5])

options_charge_line2 = {'pin_inputs': 
            {'start_pin': {'component': 'Q2', 'pin': 'Charge_Line'}, 
             'end_pin': {'component': 'Launch_CL_Q2', 'pin': 'tie'}},
            'lead': {'start_straight': '400um', 'end_straight': '100um',
                     'start_jogged_extension': jogsB_in},
            'anchors': anchors2c,
            'fillet': fillet
            }

charge_line2 = RouteAnchors(design, 'Charge_Line2', options_charge_line2)
# charge_line2 = RoutePathfinder(design, 'Charge_Line2', options_charge_line2)

anchors4c = OrderedDict()
anchors4c[0] = np.array([2, -2.5])

options_charge_line4 = {'pin_inputs': 
            {'start_pin': {'component': 'Q4', 'pin': 'Charge_Line'}, 
             'end_pin': {'component': 'Launch_CL_Q4', 'pin': 'tie'}},
            'lead': {'start_straight': '400um', 'end_straight': '225um',
                     'start_jogged_extension': jogsB_in},
            'anchors': anchors4c,
            'fillet': fillet
            }

charge_line4 = RouteAnchors(design, 'Charge_Line4', options_charge_line4)
# charge_line4 = RoutePathfinder(design, 'Charge_Line4', options_charge_line4)

gui.rebuild()
gui.autoscale()

In [ ]:
# Get a list of all the qcomponents in QDesign and then zoom on them.
all_component_names = design.components.keys()

gui.zoom_on_components(all_component_names)

In [ ]:
# Screenshot the canvas only as a .png formatted file.
# gui.screenshot()
gui.figure.savefig('shot.png')

from IPython.display import Image, display
_disp_ops = dict(width=500)
display(Image('shot.png', **_disp_ops))

In [ ]:
# # Closing the Qiskit Metal GUI
# gui.main_window.close()

In [ ]:
with open("final_four_qubit_chip_design.py", "w") as f:
    f.write(design.to_python_script())

## Chapter A
### Tuning the anharmonicity through geometry, we could target a value of 300 MHz

In [ ]:
from qiskit_metal.analyses.quantization import LOManalysis
lom_4qb = LOManalysis(design, "q3d")

In [ ]:
lom_4qb.setup

In [ ]:
# render_design_argument_qcomps = ['Q1']
# render_design_argument_endcaps = [('Q1', 'a'), ('Q1', 'b'),('Q1', 'c')]

render_design_argument_qcomps = ['Q1','Q2','Q3','Q4',
                                  'cpw1','cpw2','cpw3','cpw4',
                                  'ol1','ol2','ol3','ol4',
                                  'line_cl1','line_cl2','line_cl3','line_cl4',
                                  'P1_Q','P2_Q','P3_Q','P4_Q',
                                  'P1_C','P2_C','P3_C','P4_C']

render_design_argument_endcaps = []

In [ ]:
lom_4qb.sim.setup.max_passes = 15
lom_4qb.sim.setup.freq_ghz = 1.0
# lom_4qb.setup.freq_bus = [6.0, 6.2, 6.4, 6.6]
lom_4qb.setup.freq_bus = [6.0, 6.2, 6.4]
lom_4qb.sim.setup

In [ ]:
# sweep_data, return_code = lom_4qb.run_sweep(q1.name,
#                                        'pad_gap',
#                                        ['10um', '20um', '30um', '40um', '50um'],
#                                        render_design_argument_qcomps,
#                                        render_design_argument_endcaps,
#                                        design_name="GetCapacitance",
#                                        box_plus_buffer=True)

In [ ]:
sweep_data, return_code = lom_4qb.run_sweep(q1.name,
                                       'pad_width',
                                        # ['250um', '300um', '350um', '400um', '455um'],
                                        ['350um'],
                                       render_design_argument_qcomps,
                                       render_design_argument_endcaps,
                                       design_name="GetCapacitance",
                                       box_plus_buffer=True)

In [ ]:
first_key = list(sweep_data.keys())[0]
cap = sweep_data[first_key]['sim_variables']['cap_matrix']
print(list(cap.index))

In [ ]:
from pandas import DataFrame

ec_val = []
for opt_val in sweep_data.keys():
    ec_val.append([opt_val,sweep_data[opt_val]['variables']['lumped_oscillator']['EC']])

df=DataFrame(ec_val,columns = ['Sweep Value', 'Ec'])
df

In [ ]:
save_dir = r'C:\Users\giann\Projects\quantum-metal\qiskit-metal-simulations\results\chapter_a'
os.makedirs(save_dir, exist_ok=True)

for gap, data in sweep_data.items():
    cap_matrix = data['sim_variables']['cap_matrix']
    cap_matrix.to_csv(os.path.join(save_dir, f'cap_matrix_{gap}.csv'))

In [ ]:
cap_30um = pd.read_csv(os.path.join(save_dir, 'cap_matrix_20um.csv'), index_col=0)
cap_30um

## EJ and Lj Targets — pad_width = 350 µm

**EC = 291.12 MHz** (same for all qubits, set by pad geometry)  
**|α| ≈ 291 MHz** (same for all qubits)

| Qubit | Target fQ (GHz) | EJ (GHz) | Lj (nH) | EJ/EC |
|-------|----------------|----------|---------|-------|
| Q1    | 5.0            | 12.021   | 13.598  | 41.3  |
| Q2    | 5.5            | 14.400   | 11.352  | 49.5  |
| Q3    | 6.0            | 16.994   |  9.619  | 58.4  |
| Q4    | 6.5            | 19.802   |  8.255  | 68.0  |

### Formula used

$$f_Q \approx \sqrt{8 E_J E_C} - E_C$$

$$E_J = \frac{(f_Q + E_C)^2}{8 E_C}$$

$$L_j = \frac{(\Phi_0 / 2\pi)^2}{E_J h}$$

where $\Phi_0 = 2.068 \times 10^{-15}$ Wb.

> **Note:** Bus resonator frequencies are set to [6.0, 6.2, 6.4] GHz in the LOM setup.  
> Q3 (6.0 GHz) and Q4 (6.5 GHz) may be too close to the bus frequencies — consider shifting targets down to avoid spectral crowding.


In [ ]:
import numpy as np
from scipy.constants import h, e
import pandas as pd

# EC_MHz = sweep_data['350um']['variables']['lumped_oscillator']['EC']
# EC_MHz = 291.12
EC_MHz = 247.0
EC = EC_MHz * 1e6 # Hz
Phi0 = h / (2 * e)

target_fQ_GHz = [4.0, 4.5, 5.0, 5.5]

rows = []
for fQ_GHz in target_fQ_GHz:
    fQ = fQ_GHz * 1e9
    EJ = (fQ + EC)**2 / (8 * EC)
    Lj = (Phi0 / (2 * np.pi))**2 / (EJ * h)
    rows.append({
        'Qubit'         : f'Q{target_fQ_GHz.index(fQ_GHz)+1}',
        'fQ (GHz)'      : round(fQ_GHz, 1),
        'EJ (GHz)'      : round(EJ / 1e9, 3),
        'Lj (nH)'       : round(Lj * 1e9, 3),
        'EJ/EC'         : round(EJ / EC, 1),
    })

df = pd.DataFrame(rows)
print(f"EC  = {EC_MHz:.3f} MHz  (pad_width = 350um)")
print()
print(df.to_string(index=False))

## Part B

In [ ]:
from qiskit_metal.analyses.quantization import EPRanalysis
eig_4qb_ps = EPRanalysis(design, "hfss")

In [ ]:
pad_width = '350um'

for name, q in zip(['Q1','Q2','Q3','Q4'], [q1,q2,q3,q4]):
    q.options.pad_width = pad_width
    q.options['hfss_inductance'] = f'Lj_{name}'
    q.options['hfss_capacitance'] = f'Cj_{name}'
gui.rebuild()

In [ ]:
eig_4qb_ps.sim.renderer.options['x_buffer_width_mm'] = 1 # tune to 1.0–2.0 mm, default was 0.5
eig_4qb_ps.sim.renderer.options['y_buffer_width_mm'] = 1 # tune to 1.0–2.0 mm

In [ ]:
eig_4qb_ps.sim.setup.n_modes = 9
eig_4qb_ps.sim.setup.max_passes = 15 # passed to 2
eig_4qb_ps.sim.setup.max_delta_f = 0.05
eig_4qb_ps.sim.setup.vars = Dict(
    Lj_Q1='17.908 nH', Cj_Q1='0 fF',
    Lj_Q2='14.334 nH', Cj_Q2='0 fF',
    Lj_Q3='11.732 nH',  Cj_Q3='0 fF',
    Lj_Q4='9.780 nH',  Cj_Q4='0 fF',
)

In [ ]:
# # List every component and its pins
# for comp_name, comp in design.components.items():
#     for pin_name in comp.pins:
#         net = comp.pins[pin_name].get('net_id', None)
#         print(f"({comp_name!r}, {pin_name!r})  net={net}")

In [ ]:
# eig_4qb_ps.sim.run(
#     name="FourQubit_v2",
#     # components=['Q1','Q2','Q3','Q4','cpw1','cpw2','cpw3','cpw4'],
#     components=['Q1','Q2','Q3','Q4'],
#     open_terminations=[]
# )
# eig_4qb_ps.sim.run(
#     name="FourQubit_v5",
#     components=['Q1','Q2','Q3','Q4',
#                 'cpw1','cpw2','cpw3','cpw4',
#                 'ol1','ol2','ol3','ol4',       
#                 'line_cl1','line_cl2','line_cl3','line_cl4',
#                 'P1_C','P2_C','P3_C','P4_C',
#                 'P1_Q','P2_Q','P3_Q','P4_Q'],
#     open_terminations=[]    # launchpad geometry handles termination
# )
# eig_4qb_ps.sim.run(
#     name="FourQubit_v6",
#     components=[
#         'Q1', 'Q2', 'Q3', 'Q4',
#         'cpw1', 'cpw2', 'cpw3', 'cpw4',
#         'ol1', 'ol2', 'ol3', 'ol4',
#     ],
#     open_terminations=[
#         ('ol1', 'end'), ('ol2', 'end'),
#         ('ol3', 'end'), ('ol4', 'end'),
#     ]
# )
eig_4qb_ps.sim.run(
    name="FourQubit_v6",
    components=[
        'Q1', 'Q2', 'Q3', 'Q4',
        'cpw1', 'cpw2', 'cpw3', 'cpw4',
    ],
    open_terminations=[]
)

In [ ]:
eig_4qb_ps.sim.plot_convergences()

In [ ]:
eig_4qb_ps.get_frequencies()

In [ ]:
eig_4qb_ps.sim.print_run_args()

In [ ]:
eig_4qb_ps.sim.convergence_f

In [ ]:
eig_4qb_ps.sim.plot_fields('main')
2 3 4 8

In [ ]:
eig_4qb_ps.sim.plot_fields('main', eigenmode=9)

In [ ]:
# data_file = eig_4qb_ps.sim.renderer.epr_distributed_analysis.data_filename
# import pyEPR as epr
# eprh = epr.QuantumAnalysis(data_file)

### EPR Analysis

In [ ]:
eig_4qb_ps.setup

In [ ]:
eig_4qb_ps.del_junction()

eig_4qb_ps.add_junction('jj1', 'Lj_Q1', 'Cj_Q1', rect='JJ_rect_Lj_Q1_rect_jj', line='JJ_Lj_Q1_rect_jj_')
eig_4qb_ps.add_junction('jj2', 'Lj_Q2', 'Cj_Q2', rect='JJ_rect_Lj_Q2_rect_jj', line='JJ_Lj_Q2_rect_jj_')
eig_4qb_ps.add_junction('jj3', 'Lj_Q3', 'Cj_Q3', rect='JJ_rect_Lj_Q3_rect_jj', line='JJ_Lj_Q3_rect_jj_')
eig_4qb_ps.add_junction('jj4', 'Lj_Q4', 'Cj_Q4', rect='JJ_rect_Lj_Q4_rect_jj', line='JJ_Lj_Q4_rect_jj_')
eig_4qb_ps.setup.sweep_variable = 'variation' # 'variation'
gui.rebuild()

In [ ]:
eig_4qb_ps.setup

In [ ]:
eig_4qb_ps.setup.cos_trunc = 6 # anharmonicity α accuracy
eig_4qb_ps.setup.fock_trunc = 6  # most important reduction, higher order corrections to fQ, g, χ 4 is sufficient for fQ and g

In [ ]:
eig_4qb_ps.run_epr(modes=[0, 1, 2, 3])

In [ ]:
eig_4qb_ps.sim

In [ ]:
import pyEPR as epr
help(epr.DistributedAnalysis.do_EPR_analysis)
help(epr.QuantumAnalysis.analyze_all_variations)

In [ ]:
# eig_4qb_ps.run_epr()
eig_4qb_ps.run_epr(modes=[0, 1, 2, 3])

In [ ]:
rh = eig_4qb_ps.report_hamiltonian(eig_4qb_ps.setup.sweep_variable, numeric=True)

In [ ]:
epr_results = eig_4qb_ps.get_data()
epr_results

In [ ]:
omega_nd = eig_4qb_ps.get_frequencies()
omega_nd

In [ ]:
eig_4qb_ps.get_stored_energy()

In [ ]:
eig_4qb_ps.sim.get_data()

In [ ]:
eig_4qb_ps.sim.renderer.pinfo.design.get_object_names()

In [ ]:
eig_4qb_ps.sim.close()
# gui.main_window.close()

## In order to get Chi

In [ ]:
from pyEPR.core_quantum_analysis import QuantumAnalysis

file_name='C:\\data-pyEPR\Project7\\fourTransmons_hfss\\2025-07-31 08-52-15.npz'
# 
qa = QuantumAnalysis(file_name)  # path to results file
qa.analyze_variation('0', cos_trunc=8, fock_trunc=7, print_result=False)

res = qa.results['0']
chi_nd_matrix = res['chi_ND']
print("chi_ND (MHz):\n", chi_nd_matrix)

In [ ]:
result = qa.analyze_all_variations(cos_trunc = 8, fock_trunc = 7)
print(type(result))
list(result.items())[0]

In [ ]:
import logging
from io import StringIO

# If you're in Jupyter, try capturing IPython display output
try:
    from IPython.utils.capture import capture_output
    
    with capture_output() as captured:
        eig_4qb.report_hamiltonian(eig_4qb.setup.sweep_variable, numeric=True)
    
    print("Captured stdout:", captured.stdout)
    print("Captured stderr:", captured.stderr)
    print("Captured outputs:", captured.outputs)
except ImportError:
    print("Not in IPython/Jupyter environment")